# step-counter-increment composite — cx16: step counter feeds the wandb step kwarg

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `step-counter-increment`, `wandb-log-step`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F
import wandb

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "step-counter-increment"
DD_ATOM_IDS = ["step-counter-increment", "wandb-log-step"]
DD_SUBTOPICS = ["Trainer: step counter increment", "Logging: wandb.log step"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

The step counter and the wandb log share the same number. Increment first, log second:

```python
optimizer.step()
optimizer.zero_grad()
self.step += 1                                  # atom A.
wandb.log({'loss': loss.item()}, step=self.step)  # atom B: uses atom A's value.
```

**Atom A — `step-counter-increment`.** The counter measures committed updates. Tick AFTER `optimizer.step()` (see cx14). The value AT THE TIME OF LOG must reflect 'this many updates have happened'.

**Atom B — `wandb-log-step`.** `wandb.log(metrics, step=N)` plots `metrics` at x=N on the dashboard. If `step` is omitted, wandb uses an internal monotonic counter — for single-runs that's fine, but it makes runs with different log frequencies uncomparable.

**The composition.** Log AFTER increment — so the FIRST log entry has step=1, not step=0. If you log BEFORE incrementing, step=0 means 'before any update' but the metrics you're logging are post-update. That mismatch is the off-by-one bug that every trainer hits at least once. Test by inspecting `wandb.log.call_args_list` and asserting `[1, 2, 3, ...]`.

### Composite Exercise — step counter feeds the wandb step kwarg

**Atoms exercised together**: `step-counter-increment`, `wandb-log-step`

Implement `cx16_log_loop(losses, start_step)`.

Inputs:
- `losses`: list of Python floats — one per fake training step.
- `start_step`: int — counter value BEFORE this loop.

For each `loss` in `losses`:
1. Increment counter: `step += 1` (atom A — BEFORE logging).
2. Call `wandb.log({'loss': loss}, step=step)` (atom B).

Return the final `step`.

**Test asserts** (via mocked wandb):
- One log call per loss.
- Step kwarg sequence is `start_step+1, start_step+2, ...` (off-by-one matters).
- Loss kwarg in each call equals the corresponding input loss.
- Calling twice with `start_step=final_from_call_1` produces a continuous sequence (no reset).

In [ ]:
def cx16_log_loop(losses, start_step):
    step = start_step
    for loss in losses:
        # Atom A: increment FIRST so log entry's step reflects post-update state.
        step += 1
        # Atom B: wandb.log with step kwarg = current counter value.
        wandb.log({'loss': loss}, step=step)
    return step


<details><summary>Show solution — cx16</summary>

```python
def cx16_log_loop(losses, start_step):
    step = start_step
    for loss in losses:
        # Atom A: increment FIRST so log entry's step reflects post-update state.
        step += 1
        # Atom B: wandb.log with step kwarg = current counter value.
        wandb.log({'loss': loss}, step=step)
    return step
```

Increment-then-log (not log-then-increment) is the convention because the metric being logged reflects the state AFTER `step` updates have been applied. Logging at step=0 with post-update metrics would mean 'before any update happened, here's a post-update loss' — which makes no sense. The chained-call test (Case C) is the real-world resumption pattern: epoch N+1 starts where epoch N ended.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx16'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx16',
        'subtopics': ["Trainer: step counter increment", "Logging: wandb.log step"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()